# Real GPU LLM Serving: What Breaks After the First Request?

**Guiding question:** a pretrained model answers one request. What changes when
we ask for longer answers, send several requests, or reuse the same opening?

The [mechanism notebook](inference-systems.ipynb) let you inspect saved token state
and simulate scheduling. That does not tell us how a real model server will behave.
Now keep one unchanged public pretrained checkpoint and measure actual HTTP requests
on one NVIDIA GPU. There is no fine-tuning, adapter, customer text, or cloud deployment.

**Evidence boundary:** implemented source and portable HTTP/protocol fixture tests.
The GPU experiments were **not executed on the authoring machine**. Checked-in outputs
are empty; every GPU number must come from your own run. No CPU/synthetic fallback
pretends to be a model measurement.

| What we can already do | The next complaint | Experiment |
|---|---|---|
| Get one real answer | Earlier token work is repeated | Same checkpoint, generation cache off/on |
| Save earlier work | Saved token state occupies memory | Inspect actual cache tensors as input grows |
| Fit one request | Several legal requests share finite capacity | Native context, concurrency and cache-budget probes |
| Generate another answer | Its opening may repeat prior work | Native cold/warm prefix-cache counters |

Use [setup-serving.sh](setup-serving.sh) inside Linux/WSL with a CUDA-visible NVIDIA
GPU and Python 3.10-3.12. Choose **Python (real LLM serving GPU)**. The separate
environment pins vLLM 0.11.0 and Transformers 4.57.1; do not install them over the
original chapter's environment. See the [run guide](README.md) and
[handwritten companion](real-llm-serving-lab-theory.md).


In [ ]:
import sys
import json
import hashlib
import statistics
from dataclasses import asdict, replace
from datetime import datetime, timezone
from pathlib import Path

CHAPTER_DIR = next(
    folder for folder in (
        Path(globals().get("__file__", globals().get("__vsc_ipynb_file__", ""))).resolve().parent,
        Path.cwd(),
        Path.cwd() / "learning" / "11-ai-infrastructure" / "07-inference-systems",
    ) if (folder / "serving_experiments.py").is_file()
)
if str(CHAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(CHAPTER_DIR))
import serving_experiments as lab


## 0. Name the experiment before starting a model

Use a quiet GPU. Another workload can change clocks, free memory and waiting time.
The first check rejects unsupported hosts and mismatched package versions; it
does not manufacture reference timings.

We pin the public **SmolLM2-135M base model**, its tokenizer revision, and FP16
serving precision. Its small size makes the mechanisms approachable, not its
throughput representative of a larger model. Running the notebook downloads
public weights if they are not already cached.


In [ ]:
environment = lab.require_gpu_environment()
RUN_DIR = CHAPTER_DIR / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
BASELINE = lab.ServerConfig(kv_cache_mib=128, prefix_caching=False, max_model_len=2048, max_sequences=4)
print(json.dumps(environment, indent=2))
print("Evidence directory:", RUN_DIR)
print("Native engine settings:", asdict(BASELINE))


## 1. First prove that the endpoint actually runs the model

**Predict:** should receiving HTTP 200 alone convince you a real model generated text?

No. The context below starts the pinned vLLM process, checks its version and loaded
model, warms it with a real completion, then sends a public tokenized prompt.
The helper uses no fixture backend. Inspect its actual command and the returned
text, token counts, and stream timing.

The same context stops only the process group it started. Each profile gets a new
directory, so a rerun does not silently overwrite earlier evidence.


In [ ]:
with lab.VllmServer(BASELINE, RUN_DIR / "smoke") as client:
    print("Actual loopback endpoint:", client.base_url)
    print((RUN_DIR / "smoke" / "command.json").read_text())
    seed_tokens = client.tokenize(lab.PUBLIC_TEXT)
    smoke = lab.run_batch(client, [lab.repeat_tokens(seed_tokens, 128)], output_tokens=32, concurrency=1)

print("Generated text:", smoke.completions[0].text)
print("Input/output tokens:", smoke.completions[0].prompt_tokens, smoke.completions[0].output_tokens)
print("Client first-text milliseconds:", smoke.mean_first_text_ms)


## 2. That works, but why redo the earlier token work?

As an answer grows, later tokens need attention information from earlier tokens.
Keep that information instead of recalculating it: this is the **KV cache**.

**Predict:** with the same model and greedy decoding, should saving this work change
the answer, the time taken, both, or neither?

The next helper loads the **same real checkpoint** with Transformers, compares
`model.generate(..., use_cache=False)` against `use_cache=True`, and checks the
generated token IDs. It warms both paths, alternates their timing order, synchronizes
CUDA, then releases the model before any vLLM process starts.
Read [the implementation](serving_experiments.py) if you want to inspect those steps.


In [ ]:
hf_cache = lab.hf_cache_experiment(prompt_tokens=256, output_tokens=32)
print("Same generated token IDs:", hf_cache.outputs_equal)
print("Uncached seconds:", hf_cache.uncached_seconds)
print("Cached seconds:", hf_cache.cached_seconds)
print("Median uncached / cached:", statistics.median(hf_cache.uncached_seconds) / statistics.median(hf_cache.cached_seconds))


**Read this comparison narrowly.** It measures generation KV reuse inside the same
Transformers implementation. It is not a Transformers-versus-vLLM benchmark.
If token outputs differ, the helper stops rather than presenting a clean speedup.
There is no required speedup threshold: a real small workload can be noisy.

**New complaint:** work we kept is now occupying memory.


In [ ]:
import matplotlib.pyplot as plt

for row in hf_cache.kv_memory:
    print(row.prompt_tokens, "input tokens:", row.actual_kv_tensor_bytes, "actual KV tensor bytes")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([row.prompt_tokens for row in hf_cache.kv_memory],
        [row.actual_kv_tensor_bytes / 1024**2 for row in hf_cache.kv_memory], "o-")
ax.set(xlabel="Input tokens", ylabel="Actual KV tensor MiB", title="Saved work has a storage cost")
fig.tight_layout()
plt.show()
plt.close(fig)


## 3. The weights fit. Why might the service still run out of room?

```text
GPU memory
[model weights] + [saved token state for active requests] + [runtime workspace]
```

The first region does not grow with each prompt. The second does. Longer inputs,
longer generated outputs, and more simultaneous requests all ask the cache to
retain more token state.

The plot counts actual Transformers key/value tensor storage, not the whole process.
vLLM reserves a native pool, rounds allocations into blocks, and needs workspace too.
Its `--kv-cache-memory-bytes` is a pool budget, **not** a total GPU-process limit.

Three different limits matter:

| Limit | Question it answers |
|---|---|
| Per-sequence context length | Does one input plus its requested output fit? |
| Shared cache pool | How much saved token state can coexist? |
| Active-sequence limit | How many requests may the engine schedule together? |

The last limit does not bound the entire waiting HTTP queue. We will observe
waiting rather than assume that every extra client gets an overload rejection.


## 4. Change input length, answer length, then client concurrency

**Predict:** if every request fits individually, must twelve requests fit
simultaneously? Will more clients always increase throughput?

First vary input length while keeping one client and 128 generated tokens.
Next keep 768 input tokens and one client, but request 32, 128, then 512 output tokens.
Finally keep the same twelve 1536-token prompts and 128 output tokens, and vary only concurrency.
Prompt-prefix caching is explicitly **off** in these comparisons. This does
not disable KV reuse within each generation.

The tokenized inputs are public, deterministic workload material, not a test of
answer quality. `ignore_eos=True` makes each successful request generate exactly
the requested number of tokens so early stopping cannot masquerade as better throughput.


In [ ]:
workload = [lab.repeat_tokens(seed_tokens, 1536, offset=index) for index in range(12)]
length_results: dict[int, lab.BatchResult] = {}
output_results: dict[int, lab.BatchResult] = {}
load_results: dict[int, lab.BatchResult] = {}
with lab.VllmServer(BASELINE, RUN_DIR / "baseline") as client:
    over_limit = lab.repeat_tokens(seed_tokens, BASELINE.max_model_len - 16)
    context_error = lab.context_limit_rejection(client, over_limit, output_tokens=64)
    print("Expected context-length rejection:", context_error)
    for length in (256, 768, 1536):
        prompts = [lab.repeat_tokens(seed_tokens, length, offset=index) for index in range(12)]
        length_results[length] = lab.run_batch(client, prompts, output_tokens=128, concurrency=1)
    for output_length in (32, 128, 512):
        prompts = [lab.repeat_tokens(seed_tokens, 768, offset=index) for index in range(12)]
        output_results[output_length] = lab.run_batch(client, prompts, output_tokens=output_length, concurrency=1)
    for concurrency in (1, 2, 4, 8):
        load_results[concurrency] = lab.run_batch(client, workload, output_tokens=128, concurrency=concurrency)

for length, row in length_results.items():
    print("Input tokens", length, "first-text ms", round(row.mean_first_text_ms, 2))
for length, row in output_results.items():
    print("Output tokens", length, "request p95 ms", round(row.p95_request_ms, 2),
          "sampled cache fraction", row.sampled_max_cache_fraction)
for concurrency, row in load_results.items():
    print(concurrency, "clients:", round(row.output_tokens_per_s, 2), "output tokens/s;",
          round(row.p95_request_ms, 2), "p95 ms; sampled waiting", row.sampled_max_waiting)


An over-context request must produce the engine's context-length error; an unrelated
HTTP 400 is not accepted as proof. Legal requests competing for the shared pool
are a different experiment.

Read speed and waiting together. More completed tokens per second can coexist with
longer waits for individual users. The helper also retains native running/waiting
gauges, cache occupancy, preemption counters, and individual request timing.

**Measurement boundary:** client first-text latency is a user-visible proxy, not an
exact internal first-token timestamp; a stream chunk may contain several tokens.
Gauge peaks are sampled, not guaranteed true maxima. A p95 from twelve requests is
a small-sample observation, not a production SLO. Raw monotonic timestamps are
comparable within this benchmark process, not wall-clock dates.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(list(load_results), [row.output_tokens_per_s for row in load_results.values()], "o-")
axes[0].set(xlabel="Concurrent clients", ylabel="Completed output tokens/s")
axes[1].plot(list(load_results), [row.p95_request_ms for row in load_results.values()], "o-")
axes[1].set(xlabel="Concurrent clients", ylabel="Observed request p95 (ms)")
fig.suptitle("One model and workload: throughput is not the same as waiting time")
fig.tight_layout()
plt.show()
plt.close(fig)


## 5. What changes when we shrink only the cache pool?

**Predict:** do we necessarily get an out-of-memory crash, or might requests wait
or have some token work recomputed?

Keep the checkpoint, input IDs, output lengths, precision, context limit and
active-sequence limit fixed. Change only the native cache pool from 128 MiB to
64 MiB. Both profiles can hold one maximum-length sequence for this small model.
The engine still owns scheduling under contention.

A paused request that later needs work redone is a **preemption**. Do not require
one to happen: a scheduler may avoid it by making requests wait instead.


In [ ]:
SMALL_CACHE = replace(BASELINE, kv_cache_mib=64)
with lab.VllmServer(SMALL_CACHE, RUN_DIR / "cache-small") as client:
    small_cache = lab.run_batch(client, workload, output_tokens=128, concurrency=4)
large_cache = load_results[4]
for budget, row in ((BASELINE.kv_cache_mib, large_cache), (SMALL_CACHE.kv_cache_mib, small_cache)):
    print(budget, "MiB:", round(row.output_tokens_per_s, 2), "output tokens/s;",
          "sampled cache fraction", row.sampled_max_cache_fraction,
          "sampled waiting", row.sampled_max_waiting, "preemptions", row.preemptions)


**Your turn:** create a new run directory and change only one prompt length or
concurrency level. Predict which limit will matter, then rerun the same workload
under both budgets. Retain the actual context error or scheduling outcome; do not
raise the model's context limit merely to force a result.

This is not a guaranteed “smaller cache is slower” demonstration. If the pool was
not the binding limit, say so. If another workload shared the GPU or measurement
variation is large, repeat before attributing the difference to the pool.


## 6. These requests repeat an opening. Can we reuse that work too?

Within one answer, saved token state avoids repeating earlier generation work.
Across requests, the engine can also reuse an identical input beginning:
**prefix caching**.

Start a new engine with only that policy changed. The workload prefix below was
not used by its warm-up. Compare the first request with repeated requests, and
inspect native queried-token/hit-token counter differences.

Each request still has to generate 32 output tokens. Check both response usage
and the native engine's generated-token counter. That is different from a
gateway returning a saved complete answer without executing the model.


In [ ]:
PREFIX_CACHE = replace(BASELINE, prefix_caching=True)
prefix_prompt = lab.repeat_tokens(seed_tokens, 1024)
with lab.VllmServer(PREFIX_CACHE, RUN_DIR / "prefix-on") as client:
    prefix_results = lab.repeated_prefix(client, prefix_prompt, repeats=4)
for row in prefix_results:
    print("Request", row.request, "first-text ms", round(row.first_text_ms, 2),
          "prefix queried/hit tokens", row.prefix_queried_tokens, row.prefix_hit_tokens,
          "usage/native output tokens", row.generated_tokens, row.engine_generated_tokens)


The helper requires actual warm-prefix hits before calling reuse demonstrated.
It does not require the measured latency to improve: this small model and workload
may be dominated by other costs.

Keep three different caches separate:

| Work avoided | Name | Owner |
|---|---|---|
| Recomputing previous tokens while one answer grows | Generation KV cache | Model/runtime |
| Recomputing an identical opening in a later request | Prefix cache | Serving engine |
| Running the model at all for a previously answered request | Whole-response cache | [Gateway](../../09-llm-gateway/README.md) |

Reuse assumes a compatible model/tokenizer and matching token prefix. Public
workload reuse here is not a multi-tenant privacy or authorization policy.


## 7. Keep the evidence, not just a convincing plot

Save exactly which model, runtime, commands, hardware, workload and counters
produced the comparison. The report below is reached only after the real GPU
experiments succeed. It does not create placeholder GPU results on other hosts.
Existing reports are not overwritten.


In [ ]:
report = {
    "evidence_class": "Local measured",
    "scope": "One public pretrained checkpoint, named GPU, controlled local workload; not production capacity.",
    "environment": environment,
    "helper_sha256": hashlib.sha256(Path(lab.__file__).read_bytes()).hexdigest(),
    "lesson_sha256": hashlib.sha256((CHAPTER_DIR / "real-llm-serving-lab.py").read_bytes()).hexdigest(),
    "profiles": {"baseline": asdict(BASELINE), "small_cache": asdict(SMALL_CACHE), "prefix": asdict(PREFIX_CACHE)},
    "workload_token_ids": workload,
    "seed_token_ids": seed_tokens,
    "smoke": asdict(smoke),
    "hf_cache": asdict(hf_cache),
    "by_input_length": {str(key): asdict(value) for key, value in length_results.items()},
    "by_output_length": {str(key): asdict(value) for key, value in output_results.items()},
    "by_concurrency": {str(key): asdict(value) for key, value in load_results.items()},
    "smaller_native_pool": asdict(small_cache),
    "context_limit_rejection": context_error,
    "prefix_reuse": [asdict(row) for row in prefix_results],
}
serialized = json.dumps(report, indent=2, allow_nan=False)
with (RUN_DIR / "experiment-results.json").open("x", encoding="utf-8") as destination:
    destination.write(serialized + "\n")
print("Retained evidence:", RUN_DIR)


## What did we establish, and what is still missing?

| Opening question | Evidence to inspect |
|---|---|
| Did the API run a real model? | Pinned launch command, version/model readiness probe, actual streamed text and usage |
| Did generation caching preserve the answer? | Same-checkpoint cached/uncached token equality and measured timing |
| Does saved token state consume memory? | Actual cache tensor bytes across input lengths |
| Which limit rejected a request? | The native context-length error, not a generic HTTP failure |
| How did more clients or a smaller pool affect work? | The common workload, request records, sampled native gauges and preemption deltas |
| Was a repeated prefix reused without returning a saved whole answer? | Native hit-token deltas plus newly generated tokens on every request |

**Built for your GPU run:** real pretrained-model HTTP serving, cache timing/storage,
native context and pool limits, concurrency probes, prefix reuse and retained run identity.
**Locally fixture-tested during authoring:** HTTP streaming, error/usage validation,
metric parsing and owned-process failure cleanup. Those tests are not model benchmarks.
**Left for later:** large-model capacity, multi-GPU inference, production p99,
tenant isolation, admission policy, cloud deployment, speculative decoding and
disaggregated serving. No multiplier in another notebook establishes them.

**The next complaint:** an engine can bound active sequences while HTTP requests
continue waiting. [Systems Composition](../09-systems-composition/README.md)
decides who owns admission, queue limits and deadlines; the
[operational lab](../10-azure-operational-llm-serving/README.md) studies those
service failures with an explicitly substituted backend.
